### Cell 1 — Environment & Frontend Project Validation


This cell detects the Angular project root by finding `angular.json` and `package.json`, indexes frontend source roots, counts Angular source/test files, and verifies `npm` is available.


In [1]:
# ============================================================
# STEP 1 — AUTO-DETECT ANGULAR ROOT + SOURCE ROOTS
# ============================================================

import json
import subprocess
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

BASE_PATH = Path.cwd().resolve()
for candidate in [BASE_PATH, *BASE_PATH.parents]:
    if ((candidate / "facade").exists() and (candidate / "frontend").exists() and (candidate / "db").exists()):
        BASE_PATH = candidate
        break
print(f"Scanning inside: {BASE_PATH}")

candidates = []
for angular_file in BASE_PATH.rglob("angular.json"):
    root = angular_file.parent
    if (root / "package.json").exists():
        candidates.append(root)

if not candidates:
    raise Exception("❌ No Angular project found (angular.json + package.json).")

PROJECT_ROOT = next((c for c in candidates if c.name == "frontend"), candidates[0])
print(f"\n✅ Angular project detected at:\n{PROJECT_ROOT}")

APP_ROOT = PROJECT_ROOT / "src" / "app"
if not APP_ROOT.exists():
    raise Exception(f"❌ Missing {APP_ROOT}")

frontend_exts = (".ts", ".html", ".css", ".scss")
app_files = [p for p in APP_ROOT.rglob("*") if p.is_file() and p.suffix in frontend_exts]
spec_files = [p for p in APP_ROOT.rglob("*.spec.ts")]

print("\n[Detected source roots]")
print("App source root:")
print(" -", APP_ROOT.relative_to(PROJECT_ROOT))
print(f"\nTotal frontend source files detected: {len(app_files)}")
print(f"Total frontend test files detected: {len(spec_files)}")

print("\nSample frontend files:")
for f in sorted(app_files)[:10]:
    print("-", f.relative_to(PROJECT_ROOT))

pkg_json = json.loads((PROJECT_ROOT / "package.json").read_text(encoding="utf-8"))
print("\n[Package]")
print("name:", pkg_json.get("name"))
print("scripts.test:", pkg_json.get("scripts", {}).get("test"))

print("\n[Checking npm installation]")
result = subprocess.run(["npm", "-v"], capture_output=True, text=True)
if result.returncode != 0:
    raise Exception("❌ npm not found in PATH.")
print("npm version:", result.stdout.strip())


Scanning inside: /home/oussamaetta/Desktop/N7/Projects-S8/Projetweb

✅ Angular project detected at:
/home/oussamaetta/Desktop/N7/Projects-S8/Projetweb/frontend

[Detected source roots]
App source root:
 - src/app

Total frontend source files detected: 100
Total frontend test files detected: 25

Sample frontend files:
- src/app/a-propos/a-propos.component.css
- src/app/a-propos/a-propos.component.html
- src/app/a-propos/a-propos.component.spec.ts
- src/app/a-propos/a-propos.component.ts
- src/app/add-event/add-event.component.css
- src/app/add-event/add-event.component.html
- src/app/add-event/add-event.component.spec.ts
- src/app/add-event/add-event.component.ts
- src/app/add-recipe/add-recipe.component.css
- src/app/add-recipe/add-recipe.component.html

[Package]
name: frontend
scripts.test: ng test

[Checking npm installation]
npm version: 10.8.2


### Cell 2 — Baseline Frontend Test (Ground Truth)


This cell runs a baseline frontend test (`npm run test -- --watch=false --browsers=ChromeHeadless`) before AI edits, so later failures can be attributed to generated changes.


In [2]:
# ==========================================
# STEP 2 — BASELINE FRONTEND TEST
# ==========================================

import subprocess
from datetime import datetime

print(f"Running baseline frontend tests in: {PROJECT_ROOT}")
TEST_CMD = ["npm", "run", "test", "--", "--watch=false", "--browsers=ChromeHeadless"]
print("Command:", " ".join(TEST_CMD), "\n")

start = datetime.now()
result = subprocess.run(TEST_CMD, cwd=str(PROJECT_ROOT), capture_output=True, text=True)
end = datetime.now()

print(f"Exit code: {result.returncode}")
print(f"Duration: {(end - start).total_seconds():.2f}s")
print("\n--- STDOUT (last 100 lines) ---")
print("\n".join(result.stdout.splitlines()[-100:]))
print("\n--- STDERR (last 100 lines) ---")
print("\n".join(result.stderr.splitlines()[-100:]))

if result.returncode == 0:
    print("\n✅ Baseline frontend tests PASSED.")
else:
    print("\n❌ Baseline frontend tests FAILED.")


Running baseline frontend tests in: /home/oussamaetta/Desktop/N7/Projects-S8/Projetweb/frontend
Command: npm run test -- --watch=false --browsers=ChromeHeadless 

Exit code: 0
Duration: 8.28s

--- STDOUT (last 100 lines) ---

> frontend@0.0.0 test
> ng test --watch=false --browsers=ChromeHeadless

04 03 2026 14:50:07.770:INFO [karma-server]: Karma v6.4.4 server started at http://localhost:9876/
04 03 2026 14:50:07.771:INFO [launcher]: Launching browsers ChromeHeadless with concurrency unlimited
04 03 2026 14:50:07.775:INFO [launcher]: Starting browser ChromeHeadless
04 03 2026 14:50:08.382:INFO [Chrome Headless 145.0.0.0 (Linux x86_64)]: Connected on socket CtFClxQrAOv9BmT-AAAB with id 18772776
Chrome Headless 145.0.0.0 (Linux x86_64): Executed 0 of 25 SUCCESS (0 secs / 0 secs)
Chrome Headless 145.0.0.0 (Linux x86_64): Executed 1 of 25 SUCCESS (0 secs / 0.156 secs)
Chrome Headless 145.0.0.0 (Linux x86_64): Executed 2 of 25 SUCCESS (0 secs / 0.172 secs)
Chrome Headless 145.0.0.0 (Linux 

### Cell 3 — Minimal Frontend Context Loader (No RAG)


This cell builds a lightweight Angular context: indexes `src/app` files (`.ts/.html/.css/.scss`), selects relevant seed files by keywords, and expands by imports/template/style references.


2-step filtering:
- keyword seed
- import/template/style expansion


In [3]:
# ==========================================
# STEP 3 — MINIMAL FRONTEND CONTEXT
# ==========================================

import re
from pathlib import Path


def read_text_file(path: Path, max_chars: int = 120_000) -> str:
    if not path.exists():
        return ""
    return path.read_text(encoding="utf-8", errors="replace")[:max_chars]


context = {
    "package.json": read_text_file(PROJECT_ROOT / "package.json", max_chars=80_000),
    "angular.json": read_text_file(PROJECT_ROOT / "angular.json", max_chars=80_000),
    "tsconfig.json": read_text_file(PROJECT_ROOT / "tsconfig.json", max_chars=40_000),
}

APP_ROOT = PROJECT_ROOT / "src" / "app"
TARGET_EXTS = {".ts", ".html", ".css", ".scss"}
file_index = sorted(p for p in APP_ROOT.rglob("*") if p.is_file() and p.suffix in TARGET_EXTS)
print(f"Indexed frontend files: {len(file_index)}")


def pick_files_by_keywords(keywords, limit=12):
    kws = [k.lower() for k in keywords]
    matches = []
    for f in file_index:
        rel = str(f.relative_to(PROJECT_ROOT)).lower()
        if any(k in rel for k in kws):
            matches.append(f)
    return matches[:limit]


def resolve_ts_import(from_file: Path, imp: str):
    if not imp.startswith("."):
        return None
    base = (from_file.parent / imp).resolve()
    for c in [base.with_suffix(".ts"), base.with_suffix(".html"), base.with_suffix(".css"), base.with_suffix(".scss"), base / "index.ts"]:
        if c.exists() and c.is_file():
            return c
    return None


def expand_by_imports_and_templates(seed_files, max_extra=10):
    extra = []
    seen = set(seed_files)
    for sf in seed_files:
        txt = read_text_file(sf, max_chars=50_000)
        if sf.suffix == ".ts":
            imports = re.findall(r"import\s+[^;]*?from\s+['\"]([^'\"]+)['\"]", txt)
            for imp in imports:
                resolved = resolve_ts_import(sf, imp)
                if resolved and resolved not in seen:
                    extra.append(resolved)
                    seen.add(resolved)
                    if len(extra) >= max_extra:
                        return seed_files + extra
            templates = re.findall(r"templateUrl\s*:\s*['\"]([^'\"]+)['\"]", txt)
            for t in templates:
                cand = (sf.parent / t).resolve()
                if cand.exists() and cand not in seen:
                    extra.append(cand)
                    seen.add(cand)
                    if len(extra) >= max_extra:
                        return seed_files + extra
    return seed_files + extra


seed = pick_files_by_keywords(["recipe", "event", "forum", "auth", "routes"], limit=8)
picked_paths = expand_by_imports_and_templates(seed, max_extra=8)

print("\nPicked files (seed + expanded):")
for f in picked_paths:
    print("-", f.relative_to(PROJECT_ROOT))


Indexed frontend files: 100

Picked files (seed + expanded):
- src/app/add-event/add-event.component.css
- src/app/add-event/add-event.component.html
- src/app/add-event/add-event.component.spec.ts
- src/app/add-event/add-event.component.ts
- src/app/add-recipe/add-recipe.component.css
- src/app/add-recipe/add-recipe.component.html
- src/app/add-recipe/add-recipe.component.spec.ts
- src/app/add-recipe/add-recipe.component.ts


In [4]:
import os, sys, requests, json
from dotenv import load_dotenv
load_dotenv(override=True)

k = os.getenv("OPENROUTER_API_KEY")
b = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
m = os.getenv("MODEL_NAME", "openai/gpt-4o-mini")

print("python:", sys.executable)
print("key head/tail:", k[:12], k[-8:], "len=", len(k))
print("base:", b, "model:", m)

r = requests.post(
    f"{b}/chat/completions",
    headers={
        "Authorization": f"Bearer {k}",
        "Content-Type": "application/json",
    },
    json={
        "model": m,
        "messages": [{"role": "user", "content": "Reply exactly OK"}],
    },
    timeout=30,
)

print("status:", r.status_code)
print("body:", r.text[:500])


python: /usr/bin/python3
key head/tail: sk-or-v1-f2b 22aa4cfc len= 73
base: https://openrouter.ai/api/v1 model: openai/gpt-4o-mini
status: 200
body: {"id":"gen-1772632221-UyLuVV19NjabkQEoxRM9","object":"chat.completion","created":1772632221,"model":"openai/gpt-4o-mini","provider":"OpenAI","system_fingerprint":"fp_373a14eb6f","choices":[{"index":0,"logprobs":null,"finish_reason":"stop","native_finish_reason":"stop","message":{"role":"assistant","content":"OK","refusal":null,"reasoning":null}}],"usage":{"prompt_tokens":10,"completion_tokens":1,"total_tokens":11,"cost":0.0000021,"is_byok":false,"prompt_tokens_details":{"cached_tokens":0,"cache_


### Cell 4 — Plan-only LLM (No Code Writing)


This cell asks the LLM for a strict JSON implementation plan for Angular changes only (files to create/modify, tests, UI contract, risks), before generating code.


In [6]:
# Cell 4b — safe plan generation (no ChatPromptTemplate)

import os, json
from langchain_openai import ChatOpenAI

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "openai/gpt-4o-mini")

llm = ChatOpenAI(
    model=MODEL_NAME,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0.2,
    max_tokens=900,
)
OPENROUTER_API_KEY
FEATURE_REQUEST = """
Add a minimal proof-of-concept Angular page to validate the frontend AI pipeline.
Route: /ai/poc
Display status and timestamp.
Frontend only, no backend changes.
"""

picked_files = {}
for p in picked_paths[:14]:
    picked_files[str(p.relative_to(PROJECT_ROOT))] = read_text_file(p, max_chars=8000)

ALLOWED_EXTS = (".ts", ".html", ".css", ".scss", ".json")
def normalize_target_path(p): return str(p).replace("\\", "/").strip().lstrip("./")
def is_allowed_frontend_target(path):
    p = normalize_target_path(path)
    return p.startswith("src/") and p.endswith(ALLOWED_EXTS) and "node_modules" not in p

schema_json = json.dumps({
    "summary": "...",
    "files_to_modify": ["..."],
    "files_to_create": ["..."],
    "tests_to_add": [{"type":"unit|component|integration","target":"...","notes":"..."}],
    "ui_contract": {"route":"/...","states":["..."],"main_elements":["..."]},
    "acceptance_checks": ["npm run test -- --watch=false --browsers=ChromeHeadless"],
    "risks": ["..."]
}, ensure_ascii=False, indent=2)

human_prompt = (
    "You must plan changes for an Angular project.\n"
    "Return STRICT JSON matching this schema example:\n"
    + schema_json
    + "\n\nContext package.json:\n" + context["package.json"][:1500]
    + "\n\nContext angular.json:\n" + context["angular.json"][:1500]
    + "\n\nPicked code:\n" + json.dumps({k: v[:3000] for k, v in picked_files.items()}, ensure_ascii=False, indent=2)
    + "\n\nFeature:\n" + FEATURE_REQUEST.strip()
)

#resp = llm.invoke(human_prompt).content
#plan = json.loads(resp)

import re

resp = llm.invoke(human_prompt).content or ""
print("RAW head:", repr(resp[:200]))

# strip markdown fences if present
clean = resp.strip()
clean = re.sub(r"^```(?:json)?\s*", "", clean)
clean = re.sub(r"\s*```$", "", clean)

# try direct parse
try:
    plan = json.loads(clean)
except json.JSONDecodeError:
    # fallback: extract first JSON object block
    m = re.search(r"\{.*\}", clean, flags=re.DOTALL)
    if not m:
        raise Exception("LLM did not return JSON. Raw output:\n" + resp[:1000])
    plan = json.loads(m.group(0))



plan["files_to_create"] = [normalize_target_path(p) for p in (plan.get("files_to_create") or [])]
plan["files_to_modify"] = [normalize_target_path(p) for p in (plan.get("files_to_modify") or [])]

bad_targets = [p for p in (plan["files_to_create"] + plan["files_to_modify"]) if not is_allowed_frontend_target(p)]
if bad_targets:
    raise Exception("❌ Disallowed frontend paths: " + json.dumps(bad_targets, ensure_ascii=False))

print("✅ Plan JSON parsed.\n")
print(json.dumps(plan, indent=2, ensure_ascii=False))


RAW head: '```json\n{\n  "summary": "Add a minimal proof-of-concept Angular page to validate the frontend AI pipeline.",\n  "files_to_modify": [\n    "src/app/app-routing.module.ts"\n  ],\n  "files_to_create": [\n    "'
✅ Plan JSON parsed.

{
  "summary": "Add a minimal proof-of-concept Angular page to validate the frontend AI pipeline.",
  "files_to_modify": [
    "src/app/app-routing.module.ts"
  ],
  "files_to_create": [
    "src/app/ai-poc/ai-poc.component.ts",
    "src/app/ai-poc/ai-poc.component.html",
    "src/app/ai-poc/ai-poc.component.css",
    "src/app/ai-poc/ai-poc.component.spec.ts"
  ],
  "tests_to_add": [
    {
      "type": "component",
      "target": "src/app/ai-poc/ai-poc.component.spec.ts",
      "notes": "Test the AI POC component to ensure it renders correctly and displays the status and timestamp."
    }
  ],
  "ui_contract": {
    "route": "/ai/poc",
    "states": [
      "initial",
      "loading",
      "loaded"
    ],
    "main_elements": [
      "status mess

### Cell 5 — Generate Frontend Code + Tests (Dry Run, No Saving)


The LLM returns a patchset, then generates each file content in full (or minimal patch where requested). Generated files stay in memory (`generated_files`) until Cell 6 applies them.


In [7]:
# ==========================================
# STEP 5 — GENERATE FILES (DRY RUN)
# ==========================================

import os
import json
import re

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

if "plan" not in globals() or not isinstance(plan, dict):
    raise Exception("❌ Run Cell 4 first.")
if "generated_files" not in globals():
    generated_files = {}

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "openai/gpt-4o-mini")
llm = ChatOpenAI(model=MODEL_NAME, api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL, temperature=0.1, max_tokens=1400)


def normalize_target_path(p):
    return str(p).replace("\\", "/").strip().lstrip("./")

def strip_fences(s: str) -> str:
    s = s.lstrip()
    s = re.sub(r"```[a-zA-Z0-9_-]*\s*\n", "", s)
    s = s.replace("\n```", "\n")
    return s.strip() + "\n"

files_to_create = [normalize_target_path(p) for p in (plan.get("files_to_create") or [])]
files_to_modify = [normalize_target_path(p) for p in (plan.get("files_to_modify") or [])]
targets = [("create", p) for p in files_to_create] + [("modify", p) for p in files_to_modify]
targets = sorted(targets, key=lambda t: (0 if t[1].endswith('.spec.ts') else 1, t[1]))[:8]

print("Targets (tests first):")
for action, p in targets:
    print("-", action, p)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a senior Angular engineer. Output ONLY raw file content. No markdown."),
    ("human", """Generate FULL content for one file.
PATH: {path}
ACTION: {action}
Repo context:
{repo_ctx}
Existing file:
{existing}
Plan JSON:
{plan_json}
""")
])

repo_ctx = {k: (v or "")[:2500] for k, v in list(picked_files.items())[:14]}
new_generated = {}
for action, path in targets:
    abs_path = PROJECT_ROOT / path
    existing = abs_path.read_text(encoding="utf-8", errors="replace")[:9000] if abs_path.exists() else ""
    msg = prompt.format_messages(path=path, action=action, repo_ctx=json.dumps(repo_ctx, ensure_ascii=False, indent=2), existing=existing, plan_json=json.dumps(plan, ensure_ascii=False, indent=2))
    content = strip_fences(llm.invoke(msg).content)
    new_generated[path] = content
    print(f"✅ Prepared {path} (chars={len(content)})")

generated_files.update(new_generated)
print("\nDone. Files stored in memory (generated_files).")
print("Next: run Cell 6.")


Targets (tests first):
- create src/app/ai-poc/ai-poc.component.spec.ts
- create src/app/ai-poc/ai-poc.component.css
- create src/app/ai-poc/ai-poc.component.html
- create src/app/ai-poc/ai-poc.component.ts
- modify src/app/app-routing.module.ts
✅ Prepared src/app/ai-poc/ai-poc.component.spec.ts (chars=1301)
✅ Prepared src/app/ai-poc/ai-poc.component.css (chars=374)
✅ Prepared src/app/ai-poc/ai-poc.component.html (chars=400)
✅ Prepared src/app/ai-poc/ai-poc.component.ts (chars=839)
✅ Prepared src/app/app-routing.module.ts (chars=712)

Done. Files stored in memory (generated_files).
Next: run Cell 6.


### Cell 6 — Safe Apply + Frontend Tests + Rollback


In [31]:
# ==========================================
# STEP 6 — APPLY + TEST + ROLLBACK
# ==========================================

import json
import shutil
import subprocess
from datetime import datetime

if "generated_files" not in globals() or len(generated_files) == 0:
    raise Exception("❌ No generated_files. Run Cell 5 first.")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_DIR = PROJECT_ROOT / ".ai_backups" / RUN_ID
ARTIFACT_DIR = PROJECT_ROOT / ".ai_artifacts" / RUN_ID
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

overwritten, created = [], []
for rel_path, content in generated_files.items():
    target = PROJECT_ROOT / rel_path
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        backup = BACKUP_DIR / rel_path
        backup.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(target, backup)
        overwritten.append(rel_path)
    else:
        created.append(rel_path)
    target.write_text(content, encoding="utf-8")

print(f"Backup directory: {BACKUP_DIR}")
print("Artifacts directory:", ARTIFACT_DIR)
print("Overwritten:", overwritten if overwritten else "None")
print("Created:", created if created else "None")

TEST_CMD = ["npm", "run", "test", "--", "--watch=false", "--browsers=ChromeHeadless"]
result = subprocess.run(TEST_CMD, cwd=str(PROJECT_ROOT), capture_output=True, text=True)

last_test_stdout = result.stdout
last_test_stderr = result.stderr
last_test_code = result.returncode

print("\nExit code:", last_test_code)
print("\n--- STDOUT (last 80 lines) ---")
print("\n".join(last_test_stdout.splitlines()[-80:]))
print("\n--- STDERR (last 80 lines) ---")
print("\n".join(last_test_stderr.splitlines()[-80:]))

(ARTIFACT_DIR / "meta.json").write_text(json.dumps({"run_id": RUN_ID, "overwritten": overwritten, "created": created, "test_exit_code": last_test_code}, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "test_stdout.txt").write_text(last_test_stdout, encoding="utf-8")
(ARTIFACT_DIR / "test_stderr.txt").write_text(last_test_stderr, encoding="utf-8")

if last_test_code != 0:
    print("\n❌ Frontend tests failed. Rolling back...")
    for rel_path in overwritten:
        shutil.copy2(BACKUP_DIR / rel_path, PROJECT_ROOT / rel_path)
    for rel_path in created:
        t = PROJECT_ROOT / rel_path
        if t.exists():
            t.unlink()
    print("✅ Rollback done. Repo restored. Artifacts preserved.")
else:
    print("\n✅ Frontend tests PASSED. Patch kept.")


Backup directory: /home/oussamaetta/Desktop/N7/Projects-S8/Projetweb/frontend/.ai_backups/20260304_152815
Artifacts directory: /home/oussamaetta/Desktop/N7/Projects-S8/Projetweb/frontend/.ai_artifacts/20260304_152815
Overwritten: None
Created: ['src/app/ai-poc/ai-poc.component.spec.ts', 'src/app/ai-poc/ai-poc.component.css', 'src/app/ai-poc/ai-poc.component.html', 'src/app/ai-poc/ai-poc.component.ts', 'src/app/app-routing.module.ts']

Exit code: 1

--- STDOUT (last 80 lines) ---

> frontend@0.0.0 test
> ng test --watch=false --browsers=ChromeHeadless

04 03 2026 15:28:23.312:INFO [karma-server]: Karma v6.4.4 server started at http://localhost:9876/
04 03 2026 15:28:23.316:INFO [launcher]: Launching browsers ChromeHeadless with concurrency unlimited
04 03 2026 15:28:23.317:ERROR [karma-server]: Error: Found 1 load error
    at Server.<anonymous> (/home/oussamaetta/Desktop/N7/Projects-S8/Projetweb/frontend/node_modules/karma/lib/server.js:243:26)
    at Object.onceWrapper (node:events:63

Cell 7 (self-repair) uses:
1) failure signal from Cell 6 (`last_test_stdout`, `last_test_stderr`),
2) generated and picked context,
3) previous plan,
to propose a minimal corrective patchset and regenerate only necessary frontend files.


In [30]:
# ==========================================
# STEP 7 — SELF-REPAIR LOOP (FRONTEND)
# ==========================================
# new CELL7 UPDATE MARKER 2026-03-04 15:25:02

import os
import json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

if "last_test_code" not in globals():
    raise Exception("❌ Run Cell 6 first.")
if last_test_code == 0:
    print("✅ Build passing. Nothing to repair.")
    raise SystemExit

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "openai/gpt-4o-mini")
llm = ChatOpenAI(model=MODEL_NAME, api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL, temperature=0.1, max_tokens=1400)

failure_tail = "\n".join((last_test_stdout + "\n" + last_test_stderr).splitlines()[-220:])
base_ctx = {k: v[:1500] for k, v in {**picked_files, **generated_files}.items()}

prompt = ChatPromptTemplate.from_messages([
    ("system", "Output ONLY valid JSON. No markdown."),
("human", """Frontend tests failed. Propose minimal JSON patchset:
{{
  "modify":[{{"path":"...","reason":"..."}}],
  "create":[{{"path":"...","reason":"..."}}]
}}
Failure tail:
{failure_tail}
Context:
{base_ctx}

# new General repair rules (strict):
- Keep standalone component/test wiring consistent.
""")

])

raw = llm.invoke(prompt.format_messages(
    failure_tail=failure_tail,
    base_ctx=json.dumps(base_ctx, ensure_ascii=False, indent=2)
)).content

import re, json
clean = (raw or "").strip()
clean = re.sub(r"^```(?:json)?\s*", "", clean, flags=re.IGNORECASE)
clean = re.sub(r"\s*```$", "", clean)

def _parse_first_json(text: str):
    decoder = json.JSONDecoder()
    for m in re.finditer(r"[\{\[]", text):
        try:
            obj, _ = decoder.raw_decode(text[m.start():])
            return obj
        except json.JSONDecodeError:
            continue
    raise Exception("LLM did not return JSON:\n" + text[:1000])

try:
    patchset = json.loads(clean)
except json.JSONDecodeError:
    patchset = _parse_first_json(clean)

if not isinstance(patchset, dict):
    raise Exception("LLM JSON must be an object with modify/create keys")


targets = []
for it in (patchset.get("modify") or [])[:2]:
    targets.append(("modify", it["path"], it.get("reason", "")))
for it in (patchset.get("create") or [])[:1]:
    targets.append(("create", it["path"], it.get("reason", "")))

if not targets:
    raise Exception("❌ Empty repair patchset")

file_prompt = ChatPromptTemplate.from_messages([
    ("system", "Output ONLY raw file content. No markdown."),
    ("human", """Regenerate one frontend file.
PATH: {path}
ACTION: {action}
REASON: {reason}
Failure tail:
{failure_tail}
Existing:
{existing}
""")
])

new_generated = {}
for action, path, reason in targets:
    abs_path = PROJECT_ROOT / path
    existing = abs_path.read_text(encoding="utf-8", errors="replace")[:10000] if abs_path.exists() else ""
    msg = file_prompt.format_messages(path=path, action=action, reason=reason, failure_tail=failure_tail, existing=existing)
    content = llm.invoke(msg).content.strip() + "\n"
    new_generated[path] = content
    print(f"✅ Repaired: {path}")

generated_files.update(new_generated)
print("\nDone. Updated generated_files. Rerun Cell 6.")


✅ Repaired: src/app/ai-poc/ai-poc.component.ts
✅ Repaired: src/app/ai-poc/ai-poc.component.html

Done. Updated generated_files. Rerun Cell 6.
